In [57]:
from data_manager import DataManager

In [58]:
dm = DataManager()

In [59]:
player_name = "LeBron James"

player_id = dm.get_player_id(player_name)
data = dm.get_and_save_player_data(player_id)[::-1]

In [60]:
display(data)

,player_name,player_position,minutes,points,rebounds,assists,efg,fg3a,fg3m,fg3_pct,fga,fgm,fta,ft_pct,steals,blocks,date,game_id
80,LeBron James,F,38.0,17,6,8,0.455,1,0,0.000,11,5,9,0.778,1,0,2013-10-29,4725
123,LeBron James,F,36.0,25,4,13,0.647,7,4,0.571,17,9,4,0.750,0,0,2013-10-30,4724
124,LeBron James,F,42.0,26,7,6,0.605,2,1,0.500,19,11,5,0.600,2,1,2013-11-01,4723
93,LeBron James,F,34.0,25,3,5,0.750,5,3,0.600,14,9,5,0.800,1,0,2013-11-03,4722
132,LeBron James,F,36.0,35,8,8,0.675,3,1,0.333,20,13,8,1.000,0,1,2013-11-05,4721
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,LeBron James,F,36.0,27,5,14,0.429,9,0,0.000,28,12,3,1.000,0,2,2024-12-01,34972
752,LeBron James,F,30.0,10,8,4,0.250,4,0,0.000,16,4,4,0.500,0,0,2024-12-02,35474
750,LeBron James,F,29.0,29,5,8,0.694,4,1,0.250,18,12,4,1.000,1,0,2024-12-04,36725
754,LeBron James,F,43.0,39,10,11,0.680,11,6,0.545,25,14,6,0.833,2,3,2024-12-06,38639


In [61]:
display(data.columns)

Index(['player_name', 'player_position', 'minutes', 'points', 'rebounds',
       'assists', 'efg', 'fg3a', 'fg3m', 'fg3_pct', 'fga', 'fgm', 'fta',
       'ft_pct', 'steals', 'blocks', 'date', 'game_id'],
      dtype='object')

In [62]:
def prepare_player_data(player_data):
    """
    Prepares data for regression modeling for an individual player.

    Parameters:
        player_data (pd.DataFrame): DataFrame containing game-level data for one player.

    Returns:
        pd.DataFrame, pd.Series: Features and target variable.
    """
    # Ensure the data is sorted by date
    player_data = player_data.sort_values('date')

    # Define features and target
    features = player_data[['minutes', 'rebounds', 'assists', 'fg3a', 'fga', 'fta', 'steals', 'blocks', 'date']]
    target = player_data['points']

    return features, target


In [63]:
def temporal_train_test_split(data, target, test_size=0.2):
    """
    Splits the data into training and testing sets based on time order.

    Parameters:
        data (pd.DataFrame): Feature matrix including a 'date' column.
        target (pd.Series): Target variable.
        test_size (float): Proportion of data to use for the test set.

    Returns:
        X_train, X_test, y_train, y_test: Split features and targets.
    """
    # Sort data by date
    data = data.sort_values('date')

    # Calculate the split index
    split_idx = int(len(data) * (1 - test_size))

    # Split features and target
    X_train = data.iloc[:split_idx].drop(columns=['date'])
    X_test = data.iloc[split_idx:].drop(columns=['date'])
    y_train = target.iloc[:split_idx]
    y_test = target.iloc[split_idx:]

    return X_train, X_test, y_train, y_test


In [64]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

def train_player_model(features, target):
    """
    Trains a regression model for an individual player.

    Parameters:
        features (pd.DataFrame): Feature matrix.
        target (pd.Series): Target variable (e.g., points).

    Returns:
        model: Trained regression model.
        float: RMSE of the model on the test set.
    """
    # Temporal train-test split
    X_train, X_test, y_train, y_test = temporal_train_test_split(features, target, test_size=0.2)

    # Train a Random Forest Regressor
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    # Evaluate the model
    predictions = model.predict(X_test)
    rmse = np.sqrt(root_mean_squared_error(y_test, predictions))

    return model, rmse



In [65]:
def train_models_for_all_players(data):
    """
    Trains regression models for all players in the dataset.

    Parameters:
        data (pd.DataFrame): Combined DataFrame containing all players' game data.

    Returns:
        dict: Dictionary of player names and their trained models.
    """
    players = data['player_name'].unique()
    models = {}

    for player in players:
        print(f"Training model for {player}...")
        player_data = data[data['player_name'] == player]
        features, target = prepare_player_data(player_data)

        if len(features) > 0:  # Ensure there's enough data to train
            model, rmse = train_player_model(features, target)
            models[player] = {
                'model': model,
                'rmse': rmse
            }
            print(f"Model for {player} trained. RMSE: {rmse:.2f}")
        else:
            print(f"Not enough data for {player}, skipping...")

    return models


In [66]:
models = train_models_for_all_players(data)

Training model for LeBron James...
Model for LeBron James trained. RMSE: 2.29


In [67]:
import pickle

def save_models(models, output_dir='player_models'):
    """
    Saves trained models to disk.

    Parameters:
        models (dict): Dictionary of trained models for players.
        output_dir (str): Directory to save the models.
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    for player, data in models.items():
        with open(f"{output_dir}/{player.replace(' ', '_')}_model.pkl", 'wb') as f:
            pickle.dump(data['model'], f)
        print(f"Model for {player} saved.")


In [68]:
def predict_player_performance(player, features, model_dir='player_models'):
    """
    Predicts performance difference for a player.

    Parameters:
        player (str): Player's name.
        features (pd.DataFrame): Feature matrix for the upcoming game.
        model_dir (str): Directory containing saved models.

    Returns:
        np.ndarray: Predicted performance differences.
    """
    model_path = f"{model_dir}/{player.replace(' ', '_')}_model.pkl"

    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    return model.predict(features)
